# The Vinyl Decade

### How Genres, Formats, and Label Power Reshaped Recorded Music, 2016-2026

**Course:** Big Data and Cloud Computing - Lab 2  
**Authors:** Q Angos, Jon Robin San Diego  
**Date:** November 16, 2026  
**Dataset:** Discogs data dumps, snapshots of 1 January 2020 and 1 October 2026

> **Status: skeleton.** Code cells are written against the `vinyl/` modules but have not been run on the full data yet. Every result paragraph is marked TODO.

## Contents

1. [Executive summary](#1.-Executive-summary)
2. [Problem and research questions](#2.-Problem-and-research-questions)
3. [Data](#3.-Data)
4. [Methodology](#4.-Methodology)
5. [Results](#5.-Results)
   - 5.1 RQ1 - Genre and style shifts
   - 5.2 RQ2 - Format mix and the vinyl boom
   - 5.3 RQ3 - Label power
   - 5.4 Snapshot comparison
   - 5.5 RQ4 - Communities
   - 5.6 Artist clustering
6. [Discussion](#6.-Discussion)
7. [Limitations](#7.-Limitations)
8. [Conclusion](#8.-Conclusion)
9. [Dashboard](#9.-Dashboard)
10. [References](#10.-References)
- [Appendix A - Project files](#Appendix-A---Project-files)
- [Appendix B - Function reference](#Appendix-B---Function-reference)

## 1. Executive summary

TODO - write last. One short paragraph per research question, each with its headline number:

- **RQ1 (genres and styles):** TODO
- **RQ2 (formats and reissues):** TODO
- **RQ3 (label power):** TODO
- **RQ4 (communities):** TODO

## 2. Problem and research questions

The past decade reorganized recorded music. Streaming came to dominate listening while vinyl and cassettes returned as a collector market, styles kept splintering, and major labels kept absorbing independents. Most accounts of these shifts rely on sales charts or streaming counts, which track hits and say little about the long tail: what actually got pressed, by whom, in which scenes, and through which labels.

We ask:

1. **RQ1.** How did genre and style composition shift from 2016 to 2026, and has music become more fragmented?
2. **RQ2.** How did the format mix change, and how much of the vinyl boom was reissues rather than new music?
3. **RQ3.** How concentrated is release activity within the three majors' label trees, which artists, producers, and writers dominate each label's output, and how did ownership change between snapshots?
4. **RQ4.** What communities of artists, producers, and labels emerge from the data, and do they match genre tags?

## 3. Data

### 3.1 Source

[Discogs](https://www.discogs.com/data/) publishes monthly dumps of its user-built music database as four gzipped XML files: releases, masters, labels, and artists. We use two full snapshots.

| Snapshot | Role | Compressed size |
|---|---|---|
| 1 October 2026 | Main analysis | 11.7 GB |
| 1 January 2020 | Pre-pandemic comparison | 7.2 GB |

### 3.2 Scope

Releases dated **2016-2026**, and the artists, labels, and masters they link to, taken from the October 2026 dump. The January 2020 dump is used only for the snapshot comparison (section 5.4).

### 3.3 Raw files and key fields

| File | One row is | Key fields used |
|---|---|---|
| releases | One physical or digital release | `_id`, `released`, `country`, `genres`, `styles`, `formats` (name, descriptions), `labels` (id, catalog number), `artists`, `extraartists` (credits with roles), `companies` (e.g. pressing plants), `master_id` |
| masters | A group of releases of the same music | `_id`, `main_release`, `year` |
| labels | A label or company | `id`, `name`, `parentLabel`, `sublabels` |
| artists | A person or group | `id`, `name`, `members`, `groups`, `aliases` |

The full nested schemas are saved in `schemas/*.json` and are passed to Spark at read time so that it does not have to infer them.

### 3.4 Normalized tables

The nested XML is flattened once into the Parquet tables below (built in [`explorers/01_ingest_and_parse.ipynb`](explorers/01_ingest_and_parse.ipynb) with `vinyl.tables`).

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from pyspark.sql import functions as F

from vinyl import (
    config, ingest, tables,          # data
    trends, formats, labels,         # RQ1, RQ2, RQ3
    snapshots, networks, clustering, # comparison, RQ4
    viz,                             # figures
)

spark = ingest.get_spark(app_name="vinyl-decade-report")

In [ ]:
# The normalized tables and what each one holds.
pd.DataFrame({"table": list(tables.TABLES), "description": list(tables.TABLES.values())})

In [ ]:
# Load the tables of both snapshots from Parquet (fast; no XML is read here).
# The 2020 snapshot only has the tables the snapshot comparison needs.
main = {name: ingest.read_table(spark, config.MAIN_SNAPSHOT, name) for name in tables.snapshot_tables(config.MAIN_SNAPSHOT)}
base = {name: ingest.read_table(spark, config.BASE_SNAPSHOT, name) for name in tables.snapshot_tables(config.BASE_SNAPSHOT)}

In [ ]:
# Table sizes, for the data description. None: not built for that snapshot.
pd.DataFrame(
    {
        "table": name,
        "rows_2026": main[name].count() if name in main else None,
        "rows_2020": base[name].count() if name in base else None,
    }
    for name in tables.TABLES
)

> **Finding (TODO).** State the number of releases in scope, the number of linked artists and labels, and the share of releases with at least one style.

## 4. Methodology

### 4.1 Pipeline

```
 gzipped XML (8 files)                                   explorers/01
        |  read once with saved schemas (vinyl.ingest)
        v
 releases_staged (nested Parquet, 2016-2026 only)
        |  flatten (vinyl.tables)
        v
 normalized Parquet tables  ------------------------------------------+
        |                                                             |
        +--> trends      style shares, entropy, co-occurrence   explorers/02  -> 5.1
        +--> formats     format mix, reissue share              explorers/03  -> 5.2
        +--> labels      label trees, major share, HHI          explorers/04  -> 5.3
        +--> snapshots   parent changes, cataloguing lag        explorers/05  -> 5.4
        +--> networks    edge lists, centrality, Leiden         explorers/06  -> 5.5
        +--> clustering  features, node2vec, UMAP, HDBSCAN      explorers/07  -> 5.6
```

The gzipped XML is read exactly once per file. Gzip cannot be split, so each file is parsed by a single task; reading with a saved schema avoids a second full pass for schema inference. Everything downstream reads Parquet.

### 4.2 How this report is organized

This notebook is the report. It holds no analysis logic of its own: every number comes from a documented function in the `vinyl/` package, and every design choice was first worked out in an explorer notebook.

| Report section | Explorer notebook | Module | What it does |
|---|---|---|---|
| 3. Data | [`01_ingest_and_parse`](explorers/01_ingest_and_parse.ipynb) | `vinyl.config`, `vinyl.ingest`, `vinyl.tables` | Reads the XML with saved schemas and writes normalized Parquet tables |
| 5.1 RQ1 | [`02_genres_and_styles`](explorers/02_genres_and_styles.ipynb) | `vinyl.trends` | Yearly style shares, Shannon entropy, style co-occurrence |
| 5.2 RQ2 | [`03_formats_and_reissues`](explorers/03_formats_and_reissues.ipynb) | `vinyl.formats` | Format shares, reissue definitions, reissue share |
| 5.3 RQ3 | [`04_labels_and_majors`](explorers/04_labels_and_majors.ipynb) | `vinyl.labels` | Label trees, major vs independent share, artist concentration |
| 5.4 Comparison | [`05_snapshot_comparison`](explorers/05_snapshot_comparison.ipynb) | `vinyl.snapshots` | Parent changes and cataloguing lag between snapshots |
| 5.5 RQ4 | [`06_networks_and_communities`](explorers/06_networks_and_communities.ipynb) | `vinyl.networks` | Edge lists, centrality, Leiden communities, agreement with genres |
| 5.6 Clustering | [`07_artist_clustering`](explorers/07_artist_clustering.ipynb) | `vinyl.clustering` | Artist features, node2vec, UMAP, HDBSCAN |
| All figures | - | `vinyl.viz` | Shared plotting helpers |

Appendix B prints the documentation of every function used.

### 4.3 Methods in brief

- **Trends (RQ1).** Share of each year's releases carrying each style; Shannon entropy of the yearly style distribution as the fragmentation measure; per-year style co-occurrence networks.
- **Formats (RQ2).** Share of releases per format by year; reissue share by year and genre. TODO: state the reissue definition chosen in explorer 03.
- **Label power (RQ3).** Label trees rebuilt from `parentLabel` links; major versus independent share by year; top-artist share and Herfindahl index of releases per artist for each label.
- **Snapshot comparison.** Labels whose parent differs between January 2020 and October 2026; share of 2016-2019 releases added only after January 2020, used to estimate how undercounted 2025-2026 are.
- **Networks (RQ4).** Artist collaboration, producer/engineer credit, and artist-label graphs; centrality measures; Leiden community detection.
- **Clustering.** Artist feature vectors (style TF-IDF, format mix, career span, label count) combined with node2vec graph embeddings, reduced with UMAP, clustered with HDBSCAN.

TODO: record the parameter values decided in each explorer's "Decisions to carry into main" list.

## 5. Results

### 5.1 RQ1 - Genre and style shifts

*How did genre and style composition shift from 2016 to 2026, and has music become more fragmented?*

Exploration and parameter choices: [`explorers/02_genres_and_styles.ipynb`](explorers/02_genres_and_styles.ipynb). Functions: `vinyl.trends`.

In [ ]:
genre_shares = trends.tag_shares_by_year(main["release_genres"], tag_col="genre").toPandas()
ax = viz.plot_lines(genre_shares, "year", "share", "genre",
                    title="TODO: finding about genre shares", ylabel="share of releases", percent=True)
viz.save_figure(ax.figure, "rq1_genre_shares")

In [ ]:
style_shares = trends.tag_shares_by_year(main["release_styles"], tag_col="style")
movers = trends.top_movers(style_shares, start_year=2016, end_year=2024, min_releases=500).toPandas()
pd.concat([movers.head(10), movers.tail(10)])   # biggest risers and fallers

> **Finding (TODO).** Which genres and styles gained and lost share, with numbers.

In [ ]:
entropy = trends.shannon_entropy_by_year(main["release_styles"]).toPandas()
ax = viz.plot_lines(entropy, "year", "evenness", title="TODO: finding about fragmentation", ylabel="evenness (0-1)")
viz.save_figure(ax.figure, "rq1_style_evenness")
entropy

In [ ]:
cooccurrence = trends.style_cooccurrence(main["release_styles"], min_count=50)
# TODO: per-year network summary (nodes, edges, density, modularity) and one network figure.
cooccurrence.groupBy("year").agg(F.count("*").alias("n_edges"), F.avg("jaccard").alias("mean_jaccard")).orderBy("year").toPandas()

> **Finding (TODO).** Has music become more fragmented? Entropy/evenness in 2016 vs the end year, and what the co-occurrence networks add.

### 5.2 RQ2 - Format mix and the vinyl boom

*How did the format mix change, and how much of the vinyl boom was reissues rather than new music?*

Exploration and parameter choices: [`explorers/03_formats_and_reissues.ipynb`](explorers/03_formats_and_reissues.ipynb). Functions: `vinyl.formats`.

In [ ]:
format_shares = formats.format_shares_by_year(main["release_formats"]).toPandas()
shown = ["Vinyl", "CD", "Cassette", "File"]   # TODO: confirm in explorer 03
ax = viz.plot_lines(format_shares[format_shares.format_name.isin(shown)], "year", "share", "format_name",
                    title="TODO: finding about the format mix", ylabel="share of releases", percent=True)
viz.save_figure(ax.figure, "rq2_format_shares")

> **Finding (TODO).** How vinyl, CD, cassette, and digital shares moved.

In [ ]:
flagged = formats.add_master_reissue_flag(main["release_formats"], main["releases"], main["masters"])
REISSUE_FLAG = "is_reissue_any"   # TODO: definition chosen in explorer 03

vinyl_reissues = formats.reissue_share(
    flagged.where(F.col("format_name") == "Vinyl"), by=["year"], flag_col=REISSUE_FLAG
).toPandas()
long = vinyl_reissues.melt(id_vars="year", value_vars=["n_new", "n_reissues"], var_name="kind", value_name="n")
ax = viz.plot_stacked_share(long, "year", "n", "kind", title="TODO: finding about vinyl reissues")
viz.save_figure(ax.figure, "rq2_vinyl_reissue_share")
vinyl_reissues.sort_values("year")

In [ ]:
by_genre = formats.reissue_share(
    flagged.where(F.col("format_name") == "Vinyl").join(main["release_genres"].select("release_id", "genre"), "release_id"),
    by=["year", "genre"], flag_col=REISSUE_FLAG,
).toPandas()
ax = viz.plot_lines(by_genre, "year", "reissue_share", "genre",
                    title="TODO: finding about reissues by genre", ylabel="reissue share of vinyl", percent=True)
viz.save_figure(ax.figure, "rq2_vinyl_reissue_by_genre")

> **Finding (TODO).** What share of vinyl releases, and of vinyl growth, were reissues; which genres drive it.

### 5.3 RQ3 - Label power

*How concentrated is release activity within the three majors' label trees, and which artists, producers, and writers dominate each label's output?*

Exploration and parameter choices: [`explorers/04_labels_and_majors.ipynb`](explorers/04_labels_and_majors.ipynb). Functions: `vinyl.labels`.

In [ ]:
label_roots = labels.resolve_label_roots(main["labels"])
tagged_labels = labels.tag_major_labels(label_roots).cache()   # uses config.MAJOR_ROOTS (TODO: confirm names)
tagged_labels.groupBy("major").agg(F.count("*").alias("n_labels")).toPandas()

In [ ]:
major_share = labels.major_share_by_year(main["release_labels"], tagged_labels).toPandas()
ax = viz.plot_lines(major_share, "year", "share", "major",
                    title="TODO: finding about major vs independent share", ylabel="share of releases", percent=True)
viz.save_figure(ax.figure, "rq3_major_share")

> **Finding (TODO).** Share of releases inside each major's tree by year, and the trend.

In [ ]:
real_artists = main["release_artists"].where(~F.col("artist_id").isin(list(networks.PLACEHOLDER_ARTIST_IDS)))
concentration = labels.artist_concentration(main["release_labels"], real_artists, min_releases=50)
(concentration.join(tagged_labels.select("label_id", "major"), "label_id")
    .groupBy("major")
    .agg(F.count("*").alias("n_labels"),
         F.expr("percentile_approx(hhi, 0.5)").alias("median_hhi"),
         F.expr("percentile_approx(top_artist_share, 0.5)").alias("median_top_artist_share"))
    .toPandas())

In [ ]:
top_producers = labels.top_contributors(main["release_labels"], main["release_credits"], config.PRODUCTION_ROLES, top_n=5)
top_writers = labels.top_contributors(main["release_labels"], main["release_credits"], config.WRITING_ROLES, top_n=5)
# TODO: show these for a hand-picked set of flagship labels per major.
top_producers.orderBy("label_id", "rank").limit(20).toPandas()

> **Finding (TODO).** How dominated labels are by their top artists (major vs independent), and who the dominant producers and writers are.

### 5.4 Snapshot comparison

*How did ownership change between January 2020 and October 2026, and how undercounted are the most recent years?*

Exploration and parameter choices: [`explorers/05_snapshot_comparison.ipynb`](explorers/05_snapshot_comparison.ipynb). Functions: `vinyl.snapshots`.

In [ ]:
parent_changes = snapshots.label_parent_changes(base["labels"], main["labels"]).cache()
parent_changes.groupBy("change").count().orderBy(F.desc("count")).toPandas()

In [ ]:
# Largest labels (by in-scope releases) whose parent changed.
volume = main["release_labels"].groupBy("label_id").agg(F.countDistinct("release_id").alias("n_releases"))
(parent_changes.where(F.col("change").isin("changed_parent", "gained_parent", "lost_parent"))
    .join(volume, "label_id").orderBy(F.desc("n_releases")).limit(20).toPandas())

> **Finding (TODO).** Which notable labels moved into or out of a major's tree.

In [ ]:
lag = snapshots.cataloguing_lag(base["releases"], main["releases"])
lag_pd = lag.toPandas()
ax = viz.plot_lines(lag_pd[lag_pd.year <= 2019], "year", "completeness",
                    title="TODO: finding about cataloguing lag", ylabel="already catalogued by Jan 2020", percent=True)
viz.save_figure(ax.figure, "cmp_cataloguing_lag")
lag_pd

In [ ]:
yearly = main["releases"].groupBy("year").agg(F.count("*").alias("n_releases"))
snapshots.estimate_undercount(lag, yearly).toPandas()

> **Finding (TODO).** Share of 2016-2019 releases added after January 2020, and the implied undercount of 2025-2026.

### 5.5 RQ4 - Communities

*What communities of artists, producers, and labels emerge from the data, and do they match genre tags?*

Exploration and parameter choices: [`explorers/06_networks_and_communities.ipynb`](explorers/06_networks_and_communities.ipynb). Functions: `vinyl.networks`.

In [ ]:
collab_edges = networks.collaboration_edges(main["release_artists"], min_weight=2)   # TODO: thresholds from explorer 06
credit_edges = networks.credit_edges(main["release_artists"], main["release_credits"], roles=config.PRODUCTION_ROLES, min_weight=2)
artist_label_edges = networks.artist_label_edges(main["release_artists"], main["release_labels"], min_weight=2)

pd.DataFrame(
    {"graph": name, "edges": edges.count()}
    for name, edges in [("collaboration", collab_edges), ("credit", credit_edges), ("artist-label", artist_label_edges)]
)

In [ ]:
collab_graph = networks.to_igraph(collab_edges.toPandas())
centrality = networks.centrality_table(collab_graph)
names = main["artists"].select(F.col("artist_id").alias("node"), "artist_name").toPandas()
centrality.merge(names, on="node").sort_values("pagerank", ascending=False).head(15)

In [ ]:
communities = networks.leiden_communities(collab_graph, resolution=1.0, seed=42)
print("communities:", communities.community.nunique(), "| modularity:", round(communities.attrs["modularity"], 3))

In [ ]:
# Agreement between communities and each artist's dominant genre.
from pyspark.sql import Window

genre_counts = (main["release_artists"].join(main["release_genres"].select("release_id", "genre"), "release_id")
                .groupBy("artist_id", "genre").count())
rank = F.row_number().over(Window.partitionBy("artist_id").orderBy(F.desc("count"), "genre"))
dominant_genre = (genre_counts.withColumn("r", rank).where("r = 1")
                  .select(F.col("artist_id").alias("node"), "genre").toPandas())

networks.community_tag_agreement(communities, dominant_genre, tag_col="genre")

> **Finding (TODO).** How many communities, what the largest ones are, and how closely they follow genre tags (NMI, ARI, purity).

### 5.6 Artist clustering

*Do artists group into recognizable types when described by what they release and who they work with?*

Exploration and parameter choices: [`explorers/07_artist_clustering.ipynb`](explorers/07_artist_clustering.ipynb). Functions: `vinyl.clustering`.

> **Not runnable yet.** `clustering.node2vec_embeddings` and `clustering.combine_features` are stubs.

In [ ]:
profile = clustering.artist_profile(real_artists, main["release_formats"], main["release_labels"], min_releases=5)
kept_artists = real_artists.join(profile.select("artist_id"), "artist_id", "left_semi")
tfidf = clustering.style_tfidf(kept_artists, main["release_styles"], min_artists=20)

In [ ]:
kept_edges = networks.collaboration_edges(kept_artists, min_weight=2).toPandas()
embeddings = clustering.node2vec_embeddings(kept_edges, dimensions=64)                       # TODO: stub
features, artist_ids = clustering.combine_features(profile.toPandas(), tfidf.toPandas(), embeddings)   # TODO: stub

embedding_2d = clustering.reduce_umap(features, n_components=2)
cluster_labels = clustering.cluster_hdbscan(clustering.reduce_umap(features, n_components=10), min_cluster_size=50)

In [ ]:
# TODO: scatter of embedding_2d coloured by cluster_labels; table describing each cluster.

> **Finding (TODO).** Number of clusters, what defines each, the share of unclustered artists, and how clusters relate to the Leiden communities.

## 6. Discussion

TODO - tie the four answers together:

- Does fragmentation of styles (5.1) go together with the shift in formats (5.2)?
- Is the vinyl boom led by majors reissuing catalogue or by independents pressing new music (5.2 with 5.3)?
- Do the communities (5.5) cut across label ownership (5.3)?

## 7. Limitations

- **Contributor bias.** Discogs reflects what contributors catalogued, which skews toward physical media and collector scenes. Digital-only music is under-represented.
- **Two snapshots, net changes only.** The comparison shows that a label's parent changed between January 2020 and October 2026, not when, and misses changes that were reversed.
- **Cataloguing lag.** Recent years are still being filled in. TODO: quote the completeness estimate from 5.4 and state which years are affected.
- **Partial final year.** The October 2026 snapshot covers nine months of 2026.
- **Label trees depend on contributors.** "Independent" means "no link to a major's tree in Discogs". Missing parent links understate the majors.
- **Release dates are free text.** Releases without a usable year are excluded. TODO: quote how many.
- TODO: any others found during exploration.

## 8. Conclusion

TODO

## 9. Dashboard

TODO - link to the dashboard and list the views it contains. The figures saved under `figures/` by `viz.save_figure` and the aggregated tables above are its inputs.

## 10. References

- Discogs. *Discogs Data Dumps.* https://www.discogs.com/data/ (snapshots of 2020-01-01 and 2026-10-01).
- Traag, V. A., Waltman, L., and van Eck, N. J. (2019). From Louvain to Leiden: guaranteeing well-connected communities. *Scientific Reports*, 9, 5233.
- Grover, A., and Leskovec, J. (2016). node2vec: Scalable feature learning for networks. *KDD '16*.
- McInnes, L., Healy, J., and Melville, J. (2018). UMAP: Uniform Manifold Approximation and Projection for dimension reduction. arXiv:1802.03426.
- Campello, R. J. G. B., Moulavi, D., and Sander, J. (2013). Density-based clustering based on hierarchical density estimates. *PAKDD 2013*.
- TODO: sources for the industry context in section 2.

## Appendix A - Project files

```
lab2/
├── main.ipynb                          this report
├── explorers/                          working notebooks, one per report section
│   ├── 01_ingest_and_parse.ipynb         XML -> Parquet (run once, slow)
│   ├── 02_genres_and_styles.ipynb        RQ1
│   ├── 03_formats_and_reissues.ipynb     RQ2
│   ├── 04_labels_and_majors.ipynb        RQ3
│   ├── 05_snapshot_comparison.ipynb      2020 vs 2026
│   ├── 06_networks_and_communities.ipynb RQ4
│   └── 07_artist_clustering.ipynb        RQ4
├── vinyl/                              documented functions used by all notebooks
│   ├── config.py      paths, snapshots, year window, constants
│   ├── ingest.py      Spark session, schema loading, XML and Parquet I/O
│   ├── tables.py      nested XML rows -> normalized tables
│   ├── trends.py      style shares, entropy, co-occurrence
│   ├── formats.py     format mix, reissue share
│   ├── labels.py      label trees, major share, concentration
│   ├── snapshots.py   parent changes, cataloguing lag
│   ├── networks.py    edge lists, centrality, Leiden
│   ├── clustering.py  features, node2vec, UMAP, HDBSCAN
│   └── viz.py         plotting helpers
├── schemas/                            saved Spark schemas of the four raw files
├── data/                               raw/ and parquet/ (or set DISCOGS_DATA_DIR)
└── figures/                            written by viz.save_figure
```

**Run order.** `explorers/01_ingest_and_parse.ipynb` once, then this notebook. The other explorers are independent of each other.

## Appendix B - Function reference

Printed from the docstrings, so it always matches the code.

In [ ]:
import inspect

modules = [config, ingest, tables, trends, formats, labels, snapshots, networks, clustering, viz]
rows = []
for module in modules:
    for name, func in inspect.getmembers(module, inspect.isfunction):
        if func.__module__ == module.__name__ and not name.startswith("_"):
            summary = (inspect.getdoc(func) or "").split("\n")[0]
            rows.append({"module": module.__name__, "function": name, "summary": summary})

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

In [ ]:
# Full documentation of any one function:
help(trends.shannon_entropy_by_year)